# Cross-Source Attack Chain Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Multi-Source Threat Investigation
> **Data Sources:** CrowdStrikeDetections, CrowdStrikeHosts, CommonSecurityLog, AWSCloudTrail, GCPAuditLogs, OktaV2_CL, MailGuard365_Threats_CL
> **Nodes:** 6 (User, Device, IPAddress, AttackStage, CloudAccount, EmailThreat)
> **Edges:** 8 (TriggeredAlert, LoggedInFrom, ScannedBy, ExfiltratedTo, PerformedAction, ReceivedEmail, UsedIP, OwnedDevice)

Traces a multi-stage attack across 6 data sources — endpoint, network, identity, cloud (AWS + GCP), and email — linking users, devices, and IPs through shared entities to reveal the complete kill chain.

## Environment & Configuration

Configure the environment, set the workspace name, and import required libraries.

In [ ]:
# SDK compatibility check
import pkg_resources
import logging

# Lookback window — adjust based on when data was ingested
TIME_WINDOW_DAYS = 30

version = pkg_resources.get_distribution('MicrosoftSentinelGraphProvider').version
print(f"\u2705 MicrosoftSentinelGraphProvider: {version}")
print(f"Running in: Region: {spark.conf.get('spark.cluster.region')}")

logging.getLogger('sentinel_graph').setLevel(logging.INFO)
print(f"\u2699\ufe0f Logging level set to: {logging.getLevelName(logging.getLogger('sentinel_graph').level)}")

## Data Ingestion

Read source tables from the Sentinel Data Lake. These tables contain the lab's
pre-ingested attack telemetry from 6 different security products.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph import GraphSpecBuilder, Graph
from pyspark.sql.functions import (
    col, lit, expr, count, first, concat_ws, split, size,
    min as spark_min, max as spark_max, get_json_object,
    when, coalesce, lower, regexp_extract, countDistinct
)

sentinel_provider = MicrosoftSentinelProvider(spark)

# \u26a0\ufe0f REQUIRED: Set your Microsoft Sentinel workspace name
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # \u2190 Replace with your Sentinel workspace name

# =============================================================================
# Read CrowdStrike Detections — endpoint alerts with MITRE tactics
# =============================================================================
cs_det_raw = sentinel_provider.read_table('CrowdStrikeDetections', WORKSPACE_NAME)
cs_det_raw = cs_det_raw.df if hasattr(cs_det_raw, 'df') else cs_det_raw
cs_det_df = (cs_det_raw
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .select(
        'TimeGenerated', 'Name', 'SeverityName', 'Tactic', 'Technique',
        'SourceAccountUpn', 'UserName', 'Filename', 'Sha256', 'Cmdline',
        'Device'
    )
    .withColumn('CSHostname', get_json_object(col('Device'), '$.hostname'))
    .withColumn('ExternalIP', get_json_object(col('Device'), '$.external_ip'))
).persist()

# =============================================================================
# Read CrowdStrike Hosts — device inventory
# =============================================================================
cs_hosts_raw = sentinel_provider.read_table('CrowdStrikeHosts', WORKSPACE_NAME)
cs_hosts_raw = cs_hosts_raw.df if hasattr(cs_hosts_raw, 'df') else cs_hosts_raw
cs_hosts_df = (cs_hosts_raw
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .select('Hostname', 'LocalIp', 'ExternalIp', 'OsVersion', 'PlatformName', 'Status')
    .withColumnRenamed('PlatformName', 'Platform')
    .dropDuplicates(['Hostname'])
).persist()

# =============================================================================
# Read CommonSecurityLog — Palo Alto firewall (scans + exfil)
# =============================================================================
csl_raw = sentinel_provider.read_table('CommonSecurityLog', WORKSPACE_NAME)
csl_raw = csl_raw.df if hasattr(csl_raw, 'df') else csl_raw
csl_df = (csl_raw
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('DeviceVendor') == 'Palo Alto Networks')
    .select(
        'TimeGenerated', 'Activity', 'SourceIP', 'DestinationIP',
        'DestinationPort', 'SentBytes', 'ReceivedBytes',
        'SourceUserName', 'SourceHostName', 'ApplicationProtocol'
    )
).persist()

# =============================================================================
# Read AWSCloudTrail — cloud API activity
# =============================================================================
aws_raw = sentinel_provider.read_table('AWSCloudTrail', WORKSPACE_NAME)
aws_raw = aws_raw.df if hasattr(aws_raw, 'df') else aws_raw
aws_df = (aws_raw
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .select(
        'TimeGenerated', 'EventName', 'UserIdentityUserName',
        'SourceIpAddress', 'AWSRegion', 'EventSource'
    )
).persist()

# =============================================================================
# Read GCPAuditLogs — cloud infrastructure activity
# =============================================================================
gcp_raw = sentinel_provider.read_table('GCPAuditLogs', WORKSPACE_NAME)
gcp_raw = gcp_raw.df if hasattr(gcp_raw, 'df') else gcp_raw
gcp_df = (gcp_raw
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .withColumn('CallerIp', get_json_object(col('RequestMetadata'), '$.callerIp'))
    .select(
        'TimeGenerated', 'MethodName', 'PrincipalEmail',
        'CallerIp', 'ProjectId', 'ServiceName'
    )
).persist()

# =============================================================================
# Read OktaV2_CL — identity events
# =============================================================================
okta_raw = sentinel_provider.read_table('OktaV2_CL', WORKSPACE_NAME)
okta_raw = okta_raw.df if hasattr(okta_raw, 'df') else okta_raw
okta_df = (okta_raw
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .select(
        'TimeGenerated', 'EventOriginalType', 'ActorUsername',
        'SrcIpAddr', 'SrcGeoCountry', 'SrcGeoCity',
        'OriginalOutcomeResult', 'EventMessage'
    )
).persist()

# =============================================================================
# Read MailGuard365_Threats_CL — email threats
# =============================================================================
mail_raw = sentinel_provider.read_table('MailGuard365_Threats_CL', WORKSPACE_NAME)
mail_raw = mail_raw.df if hasattr(mail_raw, 'df') else mail_raw
mail_df = (mail_raw
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .select(
        'TimeGenerated', 'Subject', 'SenderAddress', 'SenderIP',
        'RecipientAddress', 'ThreatVerdict', 'ThreatConfidence',
        'AttachmentName', 'Urls'
    )
).persist()

# Validation
print(f"[DATA] CrowdStrike Detections: {cs_det_df.count()}")
print(f"[DATA] CrowdStrike Hosts: {cs_hosts_df.count()}")
print(f"[DATA] CommonSecurityLog (Palo Alto): {csl_df.count()}")
print(f"[DATA] AWSCloudTrail: {aws_df.count()}")
print(f"[DATA] GCPAuditLogs: {gcp_df.count()}")
print(f"[DATA] OktaV2_CL: {okta_df.count()}")
print(f"[DATA] MailGuard365_Threats_CL: {mail_df.count()}")

## Node & Edge DataFrames

Transform raw telemetry into graph entities (nodes) and relationships (edges).

### Nodes
| Type | Key | Source | Description |
|------|-----|--------|-------------|
| **User** | UPN | All sources | Unique user identities |
| **Device** | Hostname | CrowdStrikeHosts | Endpoint devices |
| **IPAddress** | IP | All sources | External IP addresses |
| **AttackStage** | Tactic | CrowdStrikeDetections + AWS + GCP | MITRE ATT&CK tactics |
| **CloudAccount** | User+Cloud | AWSCloudTrail + GCPAuditLogs | Cloud identities |
| **EmailThreat** | SubjectHash | MailGuard365_Threats_CL | Phishing emails |

### Edges
| Type | Source \u2192 Target | Description |
|------|-----------------|-------------|
| **TriggeredAlert** | Device \u2192 AttackStage | Endpoint detection with tactic |
| **LoggedInFrom** | User \u2192 IPAddress | Okta sign-in with geo context |
| **NetworkActivity** | IPAddress \u2192 IPAddress | Firewall traffic (scans + exfil) |
| **PerformedAction** | CloudAccount \u2192 AttackStage | Cloud API call |
| **ReceivedEmail** | User \u2192 EmailThreat | Phishing email delivered |
| **UsedIP** | User \u2192 IPAddress | User associated with IP |
| **OwnedDevice** | User \u2192 Device | User-device mapping |
| **CloudIdentity** | User \u2192 CloudAccount | Maps user to cloud identity |

In [ ]:
from pyspark.sql.functions import sha2, concat

# =============================================================================
# NODE DATAFRAMES
# =============================================================================

# --- User nodes ---
# Collect unique UPNs from CrowdStrike, Okta, and MailGuard
cs_users = cs_det_df.select(col('SourceAccountUpn').alias('UPN')).where(col('UPN').isNotNull() & (col('UPN') != ''))
okta_users = okta_df.select(col('ActorUsername').alias('UPN')).where(col('UPN').isNotNull() & (col('UPN') != ''))
mail_users = mail_df.select(col('RecipientAddress').alias('UPN')).where(col('UPN').isNotNull() & (col('UPN') != ''))

user_nodes_df = (cs_users.union(okta_users).union(mail_users)
    .distinct()
    .withColumn('nodeType', lit('User'))
)

# --- Device nodes ---
device_nodes_df = (cs_hosts_df
    .select(
        col('Hostname'),
        col('LocalIp'),
        col('OsVersion'),
        col('Platform'),
        col('Status')
    )
    .where(col('Hostname').isNotNull())
    .withColumn('nodeType', lit('Device'))
)

# --- IPAddress nodes ---
# Collect unique external IPs from all sources
cs_ips = cs_det_df.select(col('ExternalIP').alias('IP')).where(col('IP').isNotNull())
okta_ips = okta_df.select(col('SrcIpAddr').alias('IP')).where(col('IP').isNotNull())
aws_ips = aws_df.select(col('SourceIpAddress').alias('IP')).where(col('IP').isNotNull())
gcp_ips = gcp_df.select(col('CallerIp').alias('IP')).where(col('IP').isNotNull())
csl_src_ips = csl_df.select(col('SourceIP').alias('IP')).where(col('IP').isNotNull())
csl_dst_ips = csl_df.select(col('DestinationIP').alias('IP')).where(col('IP').isNotNull())
mail_ips = mail_df.select(col('SenderIP').alias('IP')).where(col('IP').isNotNull())

ip_nodes_df = (cs_ips.union(okta_ips).union(aws_ips).union(gcp_ips)
    .union(csl_src_ips).union(csl_dst_ips).union(mail_ips)
    .distinct()
    .withColumn('nodeType', lit('IPAddress'))
)

# --- AttackStage nodes ---
# MITRE tactics from CrowdStrike + cloud action categories
cs_tactics = cs_det_df.select(col('Tactic').alias('StageName')).where(col('StageName').isNotNull() & (col('StageName') != ''))
aws_stages = aws_df.select(lit('Cloud Attack (AWS)').alias('StageName'))
gcp_stages = gcp_df.select(lit('Cloud Attack (GCP)').alias('StageName'))

stage_nodes_df = (cs_tactics.union(aws_stages).union(gcp_stages)
    .distinct()
    .withColumn('nodeType', lit('AttackStage'))
)

# --- CloudAccount nodes ---
aws_accounts = (aws_df
    .select(col('UserIdentityUserName').alias('CloudUser'))
    .where(col('CloudUser').isNotNull())
    .distinct()
    .withColumn('CloudProvider', lit('AWS'))
    .withColumn('CloudAccountId', concat_ws('@', col('CloudUser'), lit('AWS')))
)
gcp_accounts = (gcp_df
    .select(col('PrincipalEmail').alias('CloudUser'))
    .where(col('CloudUser').isNotNull())
    .distinct()
    .withColumn('CloudProvider', lit('GCP'))
    .withColumn('CloudAccountId', concat_ws('@', col('CloudUser'), lit('GCP')))
)
cloud_account_nodes_df = (aws_accounts.union(gcp_accounts)
    .withColumn('nodeType', lit('CloudAccount'))
)

# --- EmailThreat nodes ---
email_threat_nodes_df = (mail_df
    .withColumn('SubjectHash', sha2(col('Subject'), 256))
    .select(
        col('SubjectHash'),
        col('Subject'),
        col('SenderAddress'),
        col('ThreatVerdict'),
        col('ThreatConfidence')
    )
    .dropDuplicates(['SubjectHash'])
    .withColumn('nodeType', lit('EmailThreat'))
)

# =============================================================================
# EDGE DATAFRAMES
# =============================================================================

# --- TriggeredAlert: Device -> AttackStage ---
triggered_alert_edges_df = (cs_det_df
    .where(col('CSHostname').isNotNull() & col('Tactic').isNotNull() & (col('Tactic') != ''))
    .groupBy('CSHostname', 'Tactic')
    .agg(
        count('*').alias('AlertCount'),
        first('SeverityName').alias('TopSeverity'),
        first('Name').alias('TopAlertName'),
        spark_min('TimeGenerated').cast('string').alias('FirstSeen'),
        spark_max('TimeGenerated').cast('string').alias('LastSeen')
    )
    .withColumn('AlertCount', col('AlertCount').cast('string'))
    .withColumnRenamed('CSHostname', 'Hostname')
    .withColumnRenamed('Tactic', 'StageName')
    .withColumn('EdgeKey', concat_ws('_', col('Hostname'), col('StageName')))
    .withColumn('edgeType', lit('TriggeredAlert'))
)

# --- LoggedInFrom: User -> IPAddress ---
logged_in_from_edges_df = (okta_df
    .where(col('ActorUsername').isNotNull() & col('SrcIpAddr').isNotNull())
    .groupBy('ActorUsername', 'SrcIpAddr')
    .agg(
        count('*').alias('LoginCount'),
        first('SrcGeoCountry').alias('Country'),
        first('SrcGeoCity').alias('City'),
        spark_min('TimeGenerated').cast('string').alias('FirstSeen'),
        spark_max('TimeGenerated').cast('string').alias('LastSeen')
    )
    .withColumn('LoginCount', col('LoginCount').cast('string'))
    .withColumnRenamed('ActorUsername', 'UPN')
    .withColumnRenamed('SrcIpAddr', 'IP')
    .withColumn('EdgeKey', concat_ws('_', col('UPN'), col('IP')))
    .withColumn('edgeType', lit('LoggedInFrom'))
)

# --- NetworkActivity: SourceIP -> DestinationIP ---
network_activity_edges_df = (csl_df
    .where(col('SourceIP').isNotNull() & col('DestinationIP').isNotNull())
    .groupBy('SourceIP', 'DestinationIP')
    .agg(
        count('*').alias('EventCount'),
        first('Activity').alias('TopActivity'),
        spark_min('TimeGenerated').cast('string').alias('FirstSeen'),
        spark_max('TimeGenerated').cast('string').alias('LastSeen')
    )
    .withColumn('EventCount', col('EventCount').cast('string'))
    .withColumnRenamed('SourceIP', 'SrcIP')
    .withColumnRenamed('DestinationIP', 'DstIP')
    .withColumn('EdgeKey', concat_ws('_', col('SrcIP'), col('DstIP')))
    .withColumn('edgeType', lit('NetworkActivity'))
)

# --- PerformedAction: CloudAccount -> AttackStage ---
aws_actions = (aws_df
    .where(col('UserIdentityUserName').isNotNull())
    .withColumn('CloudAccountId', concat_ws('@', col('UserIdentityUserName'), lit('AWS')))
    .withColumn('StageName', lit('Cloud Attack (AWS)'))
    .groupBy('CloudAccountId', 'StageName')
    .agg(
        count('*').alias('ActionCount'),
        first('EventName').alias('TopAction'),
        spark_min('TimeGenerated').cast('string').alias('FirstSeen'),
        spark_max('TimeGenerated').cast('string').alias('LastSeen')
    )
    .withColumn('ActionCount', col('ActionCount').cast('string'))
)
gcp_actions = (gcp_df
    .where(col('PrincipalEmail').isNotNull())
    .withColumn('CloudAccountId', concat_ws('@', col('PrincipalEmail'), lit('GCP')))
    .withColumn('StageName', lit('Cloud Attack (GCP)'))
    .groupBy('CloudAccountId', 'StageName')
    .agg(
        count('*').alias('ActionCount'),
        first('MethodName').alias('TopAction'),
        spark_min('TimeGenerated').cast('string').alias('FirstSeen'),
        spark_max('TimeGenerated').cast('string').alias('LastSeen')
    )
    .withColumn('ActionCount', col('ActionCount').cast('string'))
)
performed_action_edges_df = (aws_actions.union(gcp_actions)
    .withColumn('EdgeKey', concat_ws('_', col('CloudAccountId'), col('StageName')))
    .withColumn('edgeType', lit('PerformedAction'))
)

# --- ReceivedEmail: User -> EmailThreat ---
received_email_edges_df = (mail_df
    .where(col('RecipientAddress').isNotNull() & col('Subject').isNotNull())
    .withColumn('SubjectHash', sha2(col('Subject'), 256))
    .select(
        col('RecipientAddress').alias('UPN'),
        col('SubjectHash'),
        col('ThreatVerdict'),
        col('TimeGenerated').cast('string').alias('ReceivedAt')
    )
    .dropDuplicates(['UPN', 'SubjectHash'])
    .withColumn('EdgeKey', concat_ws('_', col('UPN'), col('SubjectHash')))
    .withColumn('edgeType', lit('ReceivedEmail'))
)

# --- UsedIP: User -> IPAddress ---
# From CrowdStrike (user + device external IP)
cs_user_ip = (cs_det_df
    .where(col('SourceAccountUpn').isNotNull() & col('ExternalIP').isNotNull())
    .select(col('SourceAccountUpn').alias('UPN'), col('ExternalIP').alias('IP'))
)
okta_user_ip = (okta_df
    .where(col('ActorUsername').isNotNull() & col('SrcIpAddr').isNotNull())
    .select(col('ActorUsername').alias('UPN'), col('SrcIpAddr').alias('IP'))
)
used_ip_edges_df = (cs_user_ip.union(okta_user_ip)
    .distinct()
    .withColumn('EdgeKey', concat_ws('_', col('UPN'), col('IP')))
    .withColumn('edgeType', lit('UsedIP'))
)

# --- OwnedDevice: User -> Device ---
owned_device_edges_df = (cs_det_df
    .where(col('SourceAccountUpn').isNotNull() & col('CSHostname').isNotNull())
    .select(
        col('SourceAccountUpn').alias('UPN'),
        col('CSHostname').alias('Hostname')
    )
    .distinct()
    .withColumn('EdgeKey', concat_ws('_', col('UPN'), col('Hostname')))
    .withColumn('edgeType', lit('OwnedDevice'))
)

# --- CloudIdentity: User -> CloudAccount ---
aws_identity = (aws_df
    .where(col('UserIdentityUserName').isNotNull())
    .withColumn('UPN', concat_ws('@', col('UserIdentityUserName'), lit('pkwork.onmicrosoft.com')))
    .withColumn('CloudAccountId', concat_ws('@', col('UserIdentityUserName'), lit('AWS')))
    .select('UPN', 'CloudAccountId').distinct()
)
gcp_identity = (gcp_df
    .where(col('PrincipalEmail').isNotNull())
    .withColumn('UPN', col('PrincipalEmail'))
    .withColumn('CloudAccountId', concat_ws('@', col('PrincipalEmail'), lit('GCP')))
    .select('UPN', 'CloudAccountId').distinct()
)
cloud_identity_edges_df = (aws_identity.union(gcp_identity)
    .withColumn('EdgeKey', concat_ws('_', col('UPN'), col('CloudAccountId')))
    .withColumn('edgeType', lit('CloudIdentity'))
)

# Validation
print(f"[NODES] Users: {user_nodes_df.count()}, Devices: {device_nodes_df.count()}, IPs: {ip_nodes_df.count()}")
print(f"[NODES] AttackStages: {stage_nodes_df.count()}, CloudAccounts: {cloud_account_nodes_df.count()}, EmailThreats: {email_threat_nodes_df.count()}")
print(f"[EDGES] TriggeredAlert: {triggered_alert_edges_df.count()}, LoggedInFrom: {logged_in_from_edges_df.count()}")
print(f"[EDGES] NetworkActivity: {network_activity_edges_df.count()}, PerformedAction: {performed_action_edges_df.count()}")
print(f"[EDGES] ReceivedEmail: {received_email_edges_df.count()}, UsedIP: {used_ip_edges_df.count()}")
print(f"[EDGES] OwnedDevice: {owned_device_edges_df.count()}, CloudIdentity: {cloud_identity_edges_df.count()}")

## Graph Assembly

Wire everything together using the `GraphSpecBuilder` fluent API.

In [ ]:
attack_chain_graph = (GraphSpecBuilder.start()

    # ===================== NODES (6 types) =====================

    .add_node('User')
    .from_dataframe(user_nodes_df)
    .with_columns('UPN', 'nodeType', key='UPN', display='UPN')

    .add_node('Device')
    .from_dataframe(device_nodes_df)
    .with_columns('Hostname', 'LocalIp', 'OsVersion', 'Platform', 'Status', 'nodeType',
                  key='Hostname', display='Hostname')

    .add_node('IPAddress')
    .from_dataframe(ip_nodes_df)
    .with_columns('IP', 'nodeType', key='IP', display='IP')

    .add_node('AttackStage')
    .from_dataframe(stage_nodes_df)
    .with_columns('StageName', 'nodeType', key='StageName', display='StageName')

    .add_node('CloudAccount')
    .from_dataframe(cloud_account_nodes_df)
    .with_columns('CloudAccountId', 'CloudUser', 'CloudProvider', 'nodeType',
                  key='CloudAccountId', display='CloudUser')

    .add_node('EmailThreat')
    .from_dataframe(email_threat_nodes_df)
    .with_columns('SubjectHash', 'Subject', 'SenderAddress', 'ThreatVerdict',
                  'ThreatConfidence', 'nodeType',
                  key='SubjectHash', display='Subject')

    # ===================== EDGES (8 types) =====================

    .add_edge('TriggeredAlert')
    .from_dataframe(triggered_alert_edges_df)
    .source(id_column='Hostname', node_type='Device')
    .target(id_column='StageName', node_type='AttackStage')
    .with_columns('edgeType', 'AlertCount', 'TopSeverity', 'TopAlertName',
                  'FirstSeen', 'LastSeen', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

    .add_edge('LoggedInFrom')
    .from_dataframe(logged_in_from_edges_df)
    .source(id_column='UPN', node_type='User')
    .target(id_column='IP', node_type='IPAddress')
    .with_columns('edgeType', 'LoginCount', 'Country', 'City',
                  'FirstSeen', 'LastSeen', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

    .add_edge('NetworkActivity')
    .from_dataframe(network_activity_edges_df)
    .source(id_column='SrcIP', node_type='IPAddress')
    .target(id_column='DstIP', node_type='IPAddress')
    .with_columns('edgeType', 'EventCount', 'TopActivity',
                  'FirstSeen', 'LastSeen', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

    .add_edge('PerformedAction')
    .from_dataframe(performed_action_edges_df)
    .source(id_column='CloudAccountId', node_type='CloudAccount')
    .target(id_column='StageName', node_type='AttackStage')
    .with_columns('edgeType', 'ActionCount', 'TopAction',
                  'FirstSeen', 'LastSeen', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

    .add_edge('ReceivedEmail')
    .from_dataframe(received_email_edges_df)
    .source(id_column='UPN', node_type='User')
    .target(id_column='SubjectHash', node_type='EmailThreat')
    .with_columns('edgeType', 'ThreatVerdict', 'ReceivedAt', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

    .add_edge('UsedIP')
    .from_dataframe(used_ip_edges_df)
    .source(id_column='UPN', node_type='User')
    .target(id_column='IP', node_type='IPAddress')
    .with_columns('edgeType', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

    .add_edge('OwnedDevice')
    .from_dataframe(owned_device_edges_df)
    .source(id_column='UPN', node_type='User')
    .target(id_column='Hostname', node_type='Device')
    .with_columns('edgeType', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

    .add_edge('CloudIdentity')
    .from_dataframe(cloud_identity_edges_df)
    .source(id_column='UPN', node_type='User')
    .target(id_column='CloudAccountId', node_type='CloudAccount')
    .with_columns('edgeType', 'EdgeKey',
                  key='EdgeKey', display='edgeType')

).done()

In [ ]:
# Check the schema of the graph spec
attack_chain_graph.show_schema()

## Build & Publish

Build the graph from the spec. This validates the schema, prepares the data, and publishes the graph for querying.

After building, you can:
1. Query the graph interactively in this notebook session
2. Schedule a graph job to materialize it for the team
3. Explore the materialized graph in the Defender portal under **Microsoft Sentinel -> Graphs**

In [ ]:
# Build the graph
attack_graph = Graph.build(attack_chain_graph)

## Query the Graph

Run a simple GQL query to verify the graph was built and inspect its contents.

In [ ]:
# Quick GQL query — visualize a sample of the graph
result = attack_graph.query("""
    // Visualize any graph
    MATCH (x)-[y]->(z)
    RETURN *
    LIMIT 3
""")
result.show()